This workbook is to try the clustering algorithm after we add in the country groupings based on study by Cashion 2016 and all the fishing entity that are present in the SAU database.


In [99]:
# Load libraries
import pandas as pd
from feature_engineering import transform_data
from dimension_reduction import dimension_reduction
from top_categories_per_type import plot_top_categories
from gdp_lat_cleaning import gdp_lat_cleaning
from hierarchical_clustering import plot_heatmap, plot_dendrogram

raw_path = "../data/raw/adr6921_Suppl_Excel_v2.csv"
gdp_path = "../data/raw/compiled_gdp.csv"
mean_lat_path = "../data/raw/geo_mean_location.csv"
output_path = '../data/processed/'
graph_output_path = '../results/'

In [100]:
# Read in raw data
df = pd.read_csv(raw_path, encoding_errors='replace')
gdp_per_capita = pd.read_csv(gdp_path)
mean_latitude = pd.read_csv(mean_lat_path)

# Transform data and obain final dataframe with counts of factories by country and attribute category
df = transform_data(df, False)

# Set 'country' as index
X = df.set_index('country')
# obtain features for clustering
X = X.drop(columns=['total_factories'])

In [8]:
n_components = 5

# Perform dimensionality reduction using NMF
W, H = dimension_reduction(X, n_components=n_components)

# Obtain list of countries in W
countries = W.index.tolist()

# Create with top categories contributing to each type
top_categories = plot_top_categories(H, 30, False)

top_categories

alt.FacetChart(...)

In [9]:
W.head()

,Type 1,Type 2,Type 3,Type 4,Type 5
country,,,,,
Argentina,0.0,0.099640,0.000000,0.692970,0.207390
Australia,0.0,0.000000,0.840692,0.000000,0.159308
Brazil,0.0,0.000000,1.000000,0.000000,0.000000
Canada,0.0,0.168408,0.204320,0.351054,0.276218
Chile,0.0,0.000348,0.758112,0.006893,0.234647


In [ ]:
# clean gpd and mean latitude dataframes for merging with W
gdp_lat_data = gdp_lat_cleaning(
    gdp_per_capita, mean_latitude, countries, '2024')

In [37]:
gdp_per_capita.head()

,country_name,GDP_per_capita,source,link
0,Albania,6069.439031,worldbank-2019,NaN
1,Algeria,4468.453419,worldbank-2019,NaN
2,American Samoa,12886.135950,worldbank-2019,NaN
3,Angola,2493.678844,worldbank-2019,NaN
4,Antigua & Barbuda,18896.372180,worldbank-2019,NaN


In [43]:
clean_gdp = gdp_per_capita.set_index("country_name")[['GDP_per_capita']]

In [44]:
clean_gdp.head()

,GDP_per_capita
country_name,
Albania,6069.439031
Algeria,4468.453419
American Samoa,12886.135950
Angola,2493.678844
Antigua & Barbuda,18896.372180


In [64]:
mean_latitude.head()

,geo_entity_id,geo_name,lon,lat
0,0.0,Unknown filler,NaN,NaN
1,1.0,Alaska (USA),0.000000,61.295554
2,2.0,Albania,19.171094,40.760915
3,3.0,Algeria,3.216740,36.933958
4,4.0,American Samoa (USA),-169.473750,-13.790833


In [101]:
clean_latitude = mean_latitude.set_index("geo_name")[["lat"]] 

In [102]:
clean_latitude

,lat
geo_name,
Unknown filler,NaN
Alaska (USA),61.295554
Albania,40.760915
Algeria,36.933958
American Samoa (USA),-13.790833
...,...
Fernando de Noronha (Brazil),-3.759583
St Paul and St. Peter Archipelago (Brazil),0.918889
Ogasawara Islands (Japan),25.744141


In [160]:
cashion_data = pd.read_csv('../data/raw/sau_fishing_entity_info.csv')

In [159]:
cashion_data.head()

,fishing_entity_id,name,wto_status,cashion_end_use_grouping,dedicated_fleet?,DHC_byproduct?,poor_conditions,tuna_ranching/direct_feeding/bait?,low_value_fish,cashion_type
0,200,Alaska (USA),Others,unknown,n,n,n,n,n,na
1,1,Albania,Developed,only_bait,n,y,n,n,n,2
2,2,Algeria,Others,only_bait,n,n,n,y,n,3
3,3,American Samoa,Others,only_dhc,n,n,n,n,n,3
4,4,Angola,Least Developed,dedicated_industry,y,y,y,y,n,1


In [161]:
cd = cashion_data.set_index('name')

In [162]:
cd[["fishing_entity_id"]]

,fishing_entity_id
name,
Alaska (USA),200
Albania,1
Algeria,2
American Samoa,3
Angola,4
...,...
Venezuela,190
Viet Nam,161
Wallis & Futuna Isl. (France),191


In [127]:
country_list = pd.read_csv('../data/raw/country_translations.csv')

In [128]:
countries = country_list[country_list['is_currently_used_for_reconstruction'] == True]['sau_name']

In [129]:
sau_lat_name = country_list[country_list['is_currently_used_for_reconstruction'] == True][['sau_name', 'mean_lat_geo_name']]

In [131]:
all_country = pd.DataFrame(index=countries)


In [132]:
mean_lat_path = "../data/raw/geo_mean_location.csv"
mean_latitude = pd.read_csv(mean_lat_path)
clean_latitude = mean_latitude.set_index("geo_name")[["lat"]] 
clean_latitude.head()

,lat
geo_name,
Alaska (USA),61.295554
Albania,40.760915
Algeria,36.933958
American Samoa (USA),-13.790833
Andaman & Nicobar Isl. (India),9.789049


In [133]:
# rename index of clean_latitude to match sau_lat_name
cleaned_latitude = clean_latitude.join(sau_lat_name.set_index('mean_lat_geo_name'), how='inner').set_index('sau_name')

In [134]:
combined = all_country.join(clean_gdp, how='left').join(cleaned_latitude, how='left')

In [135]:
combined

,GDP_per_capita,lat
sau_name,,
Albania,6069.439031,40.760915
Algeria,4468.453419,36.933958
American Samoa,12886.135950,-13.790833
Angola,2493.678844,-11.139923
Anguilla (UK),25614.509760,19.939333
...,...,...
Vanuatu,3207.446505,-19.009306
Venezuela,2523.112752,12.552886
Viet Nam,3440.900254,13.493825


In [140]:
sau_factory_name = country_list[country_list['is_currently_used_for_reconstruction'] == True][['sau_name', 'factory_country']]

In [144]:
translated_W = W.join(sau_factory_name.set_index('factory_country'), how='inner').set_index('sau_name')

In [147]:
combined = combined.join(translated_W, how='left').fillna(0)

In [163]:
combined.head()

,GDP_per_capita,lat,Type 1,Type 2,Type 3,Type 4,Type 5
sau_name,,,,,,,
Albania,6069.439031,40.760915,0.0,0.0,0.0,0.0,0.0
Algeria,4468.453419,36.933958,0.0,0.0,0.0,0.0,0.0
American Samoa,12886.135950,-13.790833,0.0,0.0,0.0,0.0,0.0
Angola,2493.678844,-11.139923,0.0,0.0,0.0,0.0,0.0
Anguilla (UK),25614.509760,19.939333,0.0,0.0,0.0,0.0,0.0


In [167]:
cd = cd.drop(columns=['fishing_entity_id'])
cd.head()

,wto_status,cashion_end_use_grouping,dedicated_fleet?,DHC_byproduct?,poor_conditions,tuna_ranching/direct_feeding/bait?,low_value_fish,cashion_type
name,,,,,,,,
Alaska (USA),Others,unknown,n,n,n,n,n,na
Albania,Developed,only_bait,n,y,n,n,n,2
Algeria,Others,only_bait,n,n,n,y,n,3
American Samoa,Others,only_dhc,n,n,n,n,n,3
Angola,Least Developed,dedicated_industry,y,y,y,y,n,1


In [170]:
final = combined.join(cd, how='left')

In [174]:
final[final['dedicated_fleet?']=="n"]

,GDP_per_capita,lat,Type 1,Type 2,Type 3,Type 4,Type 5,wto_status,cashion_end_use_grouping,dedicated_fleet?,DHC_byproduct?,poor_conditions,tuna_ranching/direct_feeding/bait?,low_value_fish,cashion_type
sau_name,,,,,,,,,,,,,,,
Albania,6069.439031,40.760915,0.000000,0.000000,0.000000,0.000000,0.000000,Developed,only_bait,n,y,n,n,n,2
Algeria,4468.453419,36.933958,0.000000,0.000000,0.000000,0.000000,0.000000,Others,only_bait,n,n,n,y,n,3
American Samoa,12886.135950,-13.790833,0.000000,0.000000,0.000000,0.000000,0.000000,Others,only_dhc,n,n,n,n,n,3
Anguilla (UK),25614.509760,19.939333,0.000000,0.000000,0.000000,0.000000,0.000000,Others,only_bait,n,n,n,n,n,na
Antigua & Barbuda,18896.372180,18.768083,0.000000,0.000000,0.000000,0.000000,0.000000,Developing,only_dhc,n,n,n,n,n,na
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Unknown Fishing Country,18888.000000,11.637200,0.000000,0.000000,0.000000,0.000000,0.000000,Others,only_dhc,n,n,n,n,n,na
US Virgin Isl.,38633.529890,17.654379,0.000000,0.000000,0.000000,0.000000,0.000000,Others,only_bait,n,n,n,n,n,3
Venezuela,2523.112752,12.552886,0.000000,0.000000,0.000000,0.000000,0.000000,Developing,dedicated_industry,n,y,y,y,n,2
